# Técnicas de Diseño y Validación de Modelos
## Evaluación Integradora — Parte 2

<div style="display:flex; align-items:center; gap:12px; margin: 8px 0 18px 0;">
  <a href="https://colab.research.google.com/github/GabrielaOjcius/Laboratorios-Tecnicas-Diseno-Validacion-Modelos/blob/main/Evaluacion%20integradora/F_Evaluacion%20Integradora%20-%20Parte%202%20-%20Notebook.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/>
  </a>
  <span>En Colab, antes de editar: <b>Archivo → Guardar una copia en Drive</b>.</span>
</div>

**Caso:** predicción de fallas en máquinas a partir de sensores.  
**Dataset:** AI4I 2020 Predictive Maintenance Dataset — UCI Machine Learning Repository.  
**Ejes evaluados:** evaluación final sobre test, matriz de confusión, costos de error, umbral de decisión y conclusión profesional.

---

### Propósito de esta parte

En esta segunda parte retomás el análisis de la Parte 1 y evaluás el modelo final sobre el conjunto de test. Además, estudiás cómo cambia el comportamiento del clasificador cuando se modifica el umbral de decisión, una cuestión central en sistemas de IA aplicados a mantenimiento predictivo donde los errores tienen costos asimétricos.

### Instrucciones

1. Ejecutá las celdas en orden.
2. Usá el conjunto de test **únicamente** para la evaluación final en la Sección B.
3. Interpretá la matriz de confusión en términos de costos reales del negocio.
4. Completá todas las celdas marcadas con **✏️ Respuesta**.
5. Entregá el notebook con todos los outputs visibles.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
plt.style.use("seaborn-v0_8-whitegrid")

---
## A. Preparación del caso

Se vuelven a cargar los datos para que este notebook pueda ejecutarse de forma independiente. Se aplican exactamente las mismas decisiones metodológicas de la Parte 1: exclusión de columnas con fuga de información y partición estratificada con la misma semilla.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
df = pd.read_csv(url)

target          = "Machine failure"
columnas_fuga   = ["TWF", "HDF", "PWF", "OSF", "RNF"]
columnas_descartar = ["UDI", "Product ID", target] + columnas_fuga

X = df.drop(columns=columnas_descartar)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

categoricas = X.select_dtypes(include="object").columns.tolist()
numericas   = X.select_dtypes(exclude="object").columns.tolist()

preprocesamiento = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas),
        ("num", StandardScaler(), numericas),
    ]
)

print(f"Train: {X_train.shape[0]} muestras  |  Tasa de falla: {y_train.mean():.2%}")
print(f"Test:  {X_test.shape[0]}  muestras  |  Tasa de falla: {y_test.mean():.2%}")

**Pregunta A.1**  
Recuperá brevemente la decisión metodológica de la Parte 1: ¿por qué no usamos las columnas `TWF`, `HDF`, `PWF`, `OSF` y `RNF` como predictores? ¿Qué problema específico introduce usarlas en un sistema de predicción en producción?

✏️ **Respuesta:**  
*(Escribí tu respuesta aquí)*

---
## B. Modelo final y evaluación sobre test

Se selecciona el Random Forest con pesos balanceados como modelo final de referencia. Si en la Parte 1 justificaste la elección de otro modelo, podés modificar la celda siguiente antes de ejecutarla. La evaluación final sobre test se realiza **una sola vez** y sus resultados no pueden usarse para ajustar el modelo.

In [ ]:
modelo_final = Pipeline([
    ("prep", preprocesamiento),
    ("model", RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=5,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
    )),
])

modelo_final.fit(X_train, y_train)
proba_test = modelo_final.predict_proba(X_test)[:, 1]
pred_test  = modelo_final.predict(X_test)

metricas_test = pd.DataFrame([{
    "accuracy":          accuracy_score(y_test, pred_test),
    "precision":         precision_score(y_test, pred_test, zero_division=0),
    "recall":            recall_score(y_test, pred_test),
    "roc_auc":           roc_auc_score(y_test, proba_test),
    "average_precision": average_precision_score(y_test, proba_test),
}])
print("=== EVALUACIÓN FINAL — CONJUNTO DE TEST ===")
metricas_test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred_test,
    display_labels=["Sin falla", "Con falla"],
    ax=axes[0], colorbar=False, cmap="Reds"
)
axes[0].set_title("Matriz de confusión — Conjunto de test", fontweight="bold")

nombres_metricas = ["Accuracy", "Precision", "Recall", "ROC-AUC", "Avg. Precision"]
valores = metricas_test.values[0]
colores = ["#8B1A2F" if v == max(valores) else "#2E75B6" for v in valores]
axes[1].bar(nombres_metricas, valores, color=colores, alpha=0.85)
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel("Valor")
axes[1].set_title("Métricas sobre test", fontweight="bold")
for i, v in enumerate(valores):
    axes[1].text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
print(classification_report(y_test, pred_test, target_names=["Sin falla", "Con falla"]))

**Pregunta B.1**  
Interpretá la matriz de confusión. ¿Qué tipo de error sería más costoso en mantenimiento predictivo: no detectar una falla real (falso negativo) o generar una alarma innecesaria (falso positivo)? Justificá con referencia a las consecuencias concretas en una planta industrial.

✏️ **Respuesta:**  
*(Escribí tu respuesta aquí)*

---

**Pregunta B.2**  
¿Los resultados sobre test son coherentes con los resultados de validación cruzada obtenidos en la Parte 1? ¿Qué implica esa coherencia (o su ausencia) sobre la confiabilidad del modelo?

✏️ **Respuesta:**  
*(Escribí tu respuesta aquí)*

---
## C. Umbral de decisión

El umbral por defecto es 0.5, pero en mantenimiento predictivo puede ser apropiado reducirlo: se aceptan más falsas alarmas a cambio de no perder fallas reales. La curva Precision-Recall muestra el compromiso completo para todos los umbrales posibles.

In [ ]:
umbrales = [0.10, 0.20, 0.30, 0.40, 0.50]
filas_umbral = []

for umbral in umbrales:
    pred = (proba_test >= umbral).astype(int)
    filas_umbral.append({
        "umbral":    umbral,
        "accuracy":  accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall":    recall_score(y_test, pred),
        "fpr":       (pred[(y_test == 0)]).sum() / (y_test == 0).sum(),
    })

tabla_umbrales = pd.DataFrame(filas_umbral)
print("Efecto del umbral de decisión sobre las métricas:")
tabla_umbrales

In [ ]:
precision_vals, recall_vals, thresholds = precision_recall_curve(y_test, proba_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(recall_vals, precision_vals, color="#8B1A2F", lw=2)
axes[0].fill_between(recall_vals, precision_vals, alpha=0.08, color="#8B1A2F")
axes[0].set_xlabel("Recall (sensibilidad)")
axes[0].set_ylabel("Precision (exactitud de alarmas)")
axes[0].set_title("Curva Precision-Recall", fontweight="bold")
axes[0].grid(alpha=0.3)

axes[1].plot(tabla_umbrales["umbral"], tabla_umbrales["recall"],
             "o-", color="#2E75B6", lw=2, label="Recall")
axes[1].plot(tabla_umbrales["umbral"], tabla_umbrales["precision"],
             "s--", color="#8B1A2F", lw=2, label="Precision")
axes[1].plot(tabla_umbrales["umbral"], tabla_umbrales["fpr"],
             "^:", color="#28A745", lw=2, label="Tasa de falsos positivos")
axes[1].set_xlabel("Umbral de decisión")
axes[1].set_ylabel("Valor")
axes[1].set_title("Recall, Precision y FPR por umbral", fontweight="bold")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Pregunta C.1**  
Si una falla no detectada puede detener una línea de producción durante horas, ¿qué umbral de decisión elegirías? Justificá el compromiso entre recall y tasa de falsos positivos para ese umbral específico, usando los valores de la tabla.

✏️ **Respuesta:**  
*(Escribí tu respuesta aquí)*

---

**Pregunta C.2**  
¿Modificar el umbral implica volver a entrenar el modelo? ¿Qué implica esto sobre la distinción entre la etapa de entrenamiento y la etapa de despliegue de un sistema de predicción?

✏️ **Respuesta:**  
*(Escribí tu respuesta aquí)*

---
## D. Conclusión integradora

Redactá una conclusión de entre **250 y 400 palabras** que integre los resultados de ambas partes. Incluí: modelo seleccionado y por qué, métrica principal y justificación, evidencia numérica concreta, efecto del umbral en el contexto del negocio, y al menos una limitación del análisis realizado.

✏️ **Conclusión integradora:**  
*(Escribí tu conclusión aquí)*

---
*Técnicas de Diseño y Validación de Modelos (0204) — Universidad Blas Pascal*